In [0]:
source =  spark.read.table('data.ipldata.customers2_src')
control_table = spark.read.table('data.ipldata.cust_control')
destination = spark.read.table('data.ipldata.customers2_des')
import pyspark.sql.functions as f


In [0]:
last_run = control_table.where(f.col('state')== 'SUCCESS')\
    .select(f.max('last_run_time'))\
    .collect()[0][0]
if last_run is None:
    last_run_time = '2026-04-01 00:00:00'
else:
    last_run_time = last_run.strftime("%Y-%m-%d %H:%M:%S")

In [0]:
try:
    # Expire old records
    spark.sql(f"""
        MERGE INTO data.ipldata.customers2_des t
        USING (
            SELECT *
            FROM data.ipldata.customers2_src
            WHERE updated_time > TIMESTAMP('{last_run_time}')
        ) s
        ON s.customer_id = t.customer_id
        AND t.is_active = true
        WHEN MATCHED AND (
            s.phone_number != t.phone_number OR
            s.email != t.email OR
            s.city != t.city OR
            s.name != t.name
        )
        THEN UPDATE SET
            t.is_active = false,
            t.end_time = s.updated_time
                                        """)

    # insert updated records
    spark.sql(f"""
        insert into data.ipldata.customers2_des 
        (customer_id, name, phone_number, email, city, start_time, end_time, is_active)
        select 
        s.customer_id, s.name, s.phone_number, s.email, s.city, s.updated_time, null, true
        from data.ipldata.customers2_src s
        join data.ipldata.customers2_des t
        on s.customer_id = t.customer_id
        where updated_time > TIMESTAMP('{last_run_time}') 
        and t.is_active = false
        and t.end_time = s.updated_time
        and (s.phone_number != t.phone_number 
            or s.email != t.email 
            or s.city != t.city
            or s.name != t.name);
                                    """)
    
    #insert new records
    spark.sql(f"""
        insert into data.ipldata.customers2_des 
        (customer_id, name, phone_number, email, city,start_time, end_time, is_active)
        select 
        s.customer_id, s.name, s.phone_number, s.email, s.city, s.creation_time, null, true
        from data.ipldata.customers2_src s
        left join data.ipldata.customers2_des t
        on s.customer_id = t.customer_id
        and t.is_active = true
        where t.customer_id is null
        and s.creation_time > TIMESTAMP('{last_run_time}')
                                                            """)
    
    #get the update count
    update_count = destination.filter((f.col('is_active') == False) & (f.col('end_time') >= last_run))\
        .count()
    
    #get the insert count
    insert_count = destination.filter((f.col('start_time') >= last_run))\
        .count() - update_count
    
    #update the control table
    spark.sql(f"""
        insert into data.ipldata.cust_control(last_run_time,job_name, state, insert_records, update_records) 
        values(TIMESTAMP('{last_run_time}'),'SCD Pipeline', 'SUCCESS', {insert_count}, {update_count})
                              """)

except Exception as e:
    error_msg = str(e).replace("'", " ")
    spark.sql(f"""
        insert into data.ipldata.cust_control(last_run_time,job_name, state, insert_records, update_records,error_msg) 
        values(TIMESTAMP('{last_run_time}'),'SCD Pipeline', 'FAILED', 0, 0,'{error_msg}')
                              """)